In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

In [7]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    ViolenceEvaluator,
    SexualEvaluator,
    SelfHarmEvaluator,
    HateUnfairnessEvaluator,
)

try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

In [8]:
from promptflow.tracing import start_trace
# instrument OpenAI
start_trace()

Prompt flow service has started...


In [9]:
coherence_evaluator = CoherenceEvaluator(model_config=model_config)
coherence_evaluator(
    query="What is the capital of France?", 
    response="Paris is the capital of France."
)

You can view the trace detail from the following URL:
http://127.0.0.1:23333/v1.0/ui/traces/?#collection=7-Monitoring and Tracing&uiTraceId=0x78d846e4e08e4c02f246f985410022c3
You can view the trace detail from the following URL:
http://127.0.0.1:23333/v1.0/ui/traces/?#collection=7-Monitoring and Tracing&uiTraceId=0xb8809d4d0e3d6fb6139c0f71b9065f15


{'gpt_coherence': 5.0}

In [10]:
from promptflow.tracing import trace

# trace your function
@trace
def test_coherence_evaluator():
    coherence_evaluator = CoherenceEvaluator(model_config=model_config)
    score = coherence_evaluator(
        query="What is the capital of France?", 
        response="Paris is the capital of France."
    )
    print(score)

test_coherence_evaluator()

{'gpt_coherence': 5.0}


### Configure the trace to be sent to Azure Application Insights

In [ ]:
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter, AzureMonitorTraceExporter, AzureMonitorMetricExporter
from opentelemetry.instrumentation.langchain import LangchainInstrumentor
from opentelemetry.instrumentation.logging import LoggingInstrumentor
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from opentelemetry.sdk.environment_variables import OTEL_EXPORTER_OTLP_ENDPOINT

collection_name = "llmops-workshop"
# if not isinstance(trace.get_tracer_provider(), TracerProvider):
if not APPLICATIONINSIGHTS_CONNECTION_STRING:
    raise ValueError("APPLICATIONINSIGHTS_CONNECTION_STRING environment variable is not set.")
os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"] = APPLICATIONINSIGHTS_CONNECTION_STRING
# OTEL_EXPORTER_OTLP_ENDPOINT = APPLICATIONINSIGHTS_CONNECTION_STRING
# setup_exporter_from_environ()

langchain_instrumentor = LangchainInstrumentor()
if not langchain_instrumentor.is_instrumented_by_opentelemetry:
    langchain_instrumentor.instrument()
tracer_provider = TracerProvider()
trace.set_tracer_provider(tracer_provider)
configure_azure_monitor(collection_name=collection_name)
traces_exporter = AzureMonitorTraceExporter()
trace_processor = BatchSpanProcessor(traces_exporter)
tracer_provider.add_span_processor(trace_processor)
RequestsInstrumentor().instrument()

In [ ]:
#now the trace should be collected by Azure Application Insights
test_coherence_evaluator()